# 募資專案成功機率預測（XGBoost + SHAP + 機率校準）

這份 Notebook 在 Colab 上執行完整流程：
1. 安裝套件、掛載 Google Drive、設定中文字型
2. `build_dataset.py`：把分散在各類別資料夾裡的 5 種特徵 CSV 合併成訓練資料（自動排除會洩漏成功/失敗答案的欄位）
3. `crowdfunding_model.py`：訓練 XGBoost，用 Brier Score／可靠度曲線驗證機率品質，必要時套用 Isotonic Regression 校準，並用 SHAP 解釋每個特徵的貢獻

**使用前只需要改「參數設定」那個 cell 裡的路徑，其他都不用動。**

## 1. 安裝套件

In [ ]:
!pip install -q --upgrade shap xgboost

## 2. 中文字型（避免圖表中文變成方框）

之前用 `apt-get install` 裝字型有時會失敗或裝了但 matplotlib 沒認到（Colab 換機器、
套件庫沒更新、或字型快取沒重新整理都有可能）。這裡改成直接下載字型檔、
用 `addfont()` 註冊進 matplotlib——不依賴系統套件安裝，也不用重啟 Runtime，
下載完這個 cell 跑完就能用。

In [ ]:
import os, urllib.request
import matplotlib
import matplotlib.font_manager as fm

FONT_PATH = "/content/NotoSansCJKtc-Regular.otf"
if not os.path.exists(FONT_PATH):
    FONT_URL = "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf"
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)

fm.fontManager.addfont(FONT_PATH)
FONT_NAME = fm.FontProperties(fname=FONT_PATH).get_name()
matplotlib.rcParams['font.sans-serif'] = [FONT_NAME]
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"已載入中文字型：{FONT_NAME}")

## 3. 掛載 Google Drive

你的資料結構是：
```
ZecZec_Group_Data/
├─ 遊戲_New_ZecZec_Dataset/
│   ├─ video_features_project_level_candidates.csv
│   ├─ market_price_result.csv
│   ├─ image_text_ratio.csv
│   ├─ trend_score.csv
│   └─ agent_copywriting_result.csv
├─ 時尚_New_ZecZec_Dataset/
│   └─ ...
```
掛載後不用手動搬動或改資料夾結構，`build_dataset.py` 會自動遞迴搜尋所有子資料夾，
新增更多類別的資料夾時也一樣直接放進去、不用改路徑。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 參數設定 —— **只需要改這裡**

In [ ]:
# 指向 Google Drive 裡 ZecZec_Group_Data 這個資料夾（依你實際路徑調整）
UPLOAD_DIR = "/content/drive/MyDrive/ZecZec_Group_Data"

# 輸出結果要放哪裡（會自動建立資料夾）
OUTPUT_DIR = "/content/output"

TARGET_COL = "label"        # 標籤欄位名稱（1=成功, 0=失敗）
ECE_THRESHOLD = 0.05         # 觸發 Isotonic 校準的門檻
PROJECT_INDEX = 0            # 要輸出詳細報告的測試集專案索引（第幾筆）

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. `build_dataset.py` —— 合併 5 種特徵 CSV

- 用檔名關鍵字比對，不管檔案在哪個類別子資料夾底下都會被抓到
- 只把「成功/失敗」抽成標籤 `label`，來源欄位（status／category_path／路徑字串）本身不進特徵矩陣
- `trend_score` 相關檔案目前只收錄成功案例，會被讀進來做風險提示，但**不會**進入訓練特徵（避免用「有沒有這筆資料」洩漏答案）
- 類別／平台類型會自動 one-hot 展開，新增幾個類別都不需要改程式碼

In [ ]:
# -*- coding: utf-8 -*-
"""
build_dataset.py（通用版）
=================
把「同一種類型」的特徵檔案（可能來自很多個募資類別：時尚、教育、...未來還會更多）
合併成一份可直接餵給 crowdfunding_model.py 的訓練資料。

設計原則：
  - 每一種特徵類型用「檔名關鍵字」比對，一次讀進該類型底下所有檔案再疊起來，
    所以之後不管新增幾個類別的檔案，只要檔名還是同一套命名規則，直接把新檔案
    丟進 /mnt/user-data/uploads 再重跑這支腳本就行，不必改程式碼。
  - 不再假設「只有兩種類別」，category / platform_type 一律當成文字欄位保留，
    one-hot 交給 crowdfunding_model.py 在訓練時動態展開（新類別自動生出新欄位）。

【重要：洩漏防範，邏輯不變】
  - agent_copywriting 的 `status`、market_price 的 `category_path`、
    video_features 的 `category` 都藏著成功/失敗答案，只拿來抽標籤，
    抽完後這些原始欄位本身都不會進入特徵矩陣。
  - training_dataset.csv（市場契合度 PMF 特徵來源）裡的「達標率(%)」「is_hit」
    等欄位同樣可能藏著結果，這裡只擷取 pmf_a/b/c/d 開頭的欄位，其餘一律不進特徵矩陣。
"""

import glob
import re
import pandas as pd

# UPLOAD_DIR / OUTPUT_DIR 沿用上面「參數設定」cell 裡的值
OUT_PATH = f"{OUTPUT_DIR}/merged_training_data.csv"

# 檔名關鍵字 → 這種類型的所有檔案（未來新增類別的檔案只要符合關鍵字就會自動被讀到）
# 用「詞幹」比對（不含結尾 s、不含 _result/_results 後綴差異），這樣同一種檔案
# 不管命名是單數/複數、有沒有 _result 後綴都抓得到，例如：
#   market_price_result.csv 和 market_price_results.csv 都符合 *market_price*
FILE_PATTERNS = {
    "copywriting": "*copywriting*.csv",
    "video": "*video_features*.csv",
    "price": "*market_price*.csv",
    "image_text": "*image_text_ratio*.csv",
    "trend": "*training_dataset*.csv",   # 市場契合度 PMF 特徵來源，已從 trend_scores.csv 改為這份
    "summary": "*projects_summary*.csv",
}


def find_files(pattern: str):
    """遞迴搜尋 UPLOAD_DIR 底下所有子資料夾（例如 Google Drive 同步下來的
    每個類別各自一個資料夾：遊戲_New_ZecZec_Dataset/、時尚_New_ZecZec_Dataset/…），
    只要檔名符合關鍵字，不管在哪一層都會被抓到。"""
    return sorted(glob.glob(f"{UPLOAD_DIR}/**/{pattern}", recursive=True))


def load_and_concat(pattern: str) -> pd.DataFrame:
    files = find_files(pattern)
    if not files:
        return pd.DataFrame()
    frames = [pd.read_csv(f) for f in files]
    print(f"  比對到 {len(files)} 個檔案：{[f.replace(UPLOAD_DIR + '/', '') for f in files]}")
    return pd.concat(frames, ignore_index=True)


def extract_project_code(raw: str) -> str:
    """任何格式的識別碼統一抽出最後一段基準代碼。"""
    return str(raw).split("/")[-1].strip()


def extract_label(text: str):
    """從含「成功/失敗」字樣的字串取出標籤，只用來建立 y，原欄位不進特徵。"""
    m = re.search(r"(成功|失敗)", str(text))
    return {"成功": 1, "失敗": 0}.get(m.group(1)) if m else None


def extract_category_platform(path_str: str):
    """從類似 '.../<平台類型>/<類別>/<成功|失敗>/<專案代碼>' 的路徑字串中，
    通用地抓出「平台類型」與「類別」兩個文字欄位（不假設類別只有時尚/教育）。
    找不到就回傳 (None, None)，之後留給 pandas 當缺值處理。"""
    parts = str(path_str).split("/")
    label_idx = next((i for i, p in enumerate(parts) if p in ("成功", "失敗")), None)
    if label_idx is None or label_idx < 2:
        return None, None
    category = parts[label_idx - 1]
    platform_type = parts[label_idx - 2]
    return category, platform_type


def count_comma_items(s):
    if pd.isna(s) or str(s).strip() == "":
        return 0
    return len([w for w in str(s).split(",") if w.strip()])


# ------------------------------------------------------------------
# 1. 文案說服力（欄位：agent_copywriting_result*.csv）
# ------------------------------------------------------------------
print("[1/6] 文案說服力特徵")
df_copy = load_and_concat(FILE_PATTERNS["copywriting"])
df_copy_final = pd.DataFrame()
if not df_copy.empty:
    df_copy["project_id"] = df_copy["project_id"].apply(extract_project_code)
    df_copy["label"] = df_copy["status"].map({"成功": 1, "失敗": 0})  # 標籤來源，之後不進特徵
    df_copy["feat_emotional_word_count"] = df_copy["extracted_emotional_words"].apply(count_comma_items)
    df_copy["feat_spec_word_count"] = df_copy["extracted_spec_words"].apply(count_comma_items)
    df_copy["platform_type"] = df_copy["category"]          # 群眾募資 / 預購式專案 / 未來新類型
    df_copy["category_name"] = df_copy["subcategory"]        # 時尚 / 教育 / 未來新類別

    copy_features = [
        "feat_text_story_ratio", "feat_text_spec_ratio_rule", "feat_text_risk_ratio",
        "feat_has_social_link", "feat_punct_intensity", "feat_avg_sentence_len",
        "feat_type_token_ratio", "feat_trust_score", "feat_emotional_ratio", "feat_spec_ratio",
        "feat_emotional_word_count", "feat_spec_word_count",
    ]
    df_copy_final = df_copy[["project_id", "label", "platform_type", "category_name"] + copy_features].copy()
    # status / category / subcategory / agent_advice 等原始文字或洩漏欄位不納入特徵矩陣

# ------------------------------------------------------------------
# 2. 影像吸引力：影片（video_features*.csv）
# ------------------------------------------------------------------
print("[2/6] 影片特徵")
df_video = load_and_concat(FILE_PATTERNS["video"])
df_video_final = pd.DataFrame()
if not df_video.empty:
    df_video["project_id"] = df_video["project"].apply(extract_project_code)
    video_features = [
        "videos_included", "videos_excluded_low_coverage", "total_video_duration",
        "primary_usage_scene_soft_ratio", "primary_gemini_avg_score",
        "mean_usage_scene_soft_ratio",
        "max_usage_scene_soft_ratio", "mean_gemini_avg_score",
        "duration_weighted_usage_scene_ratio",
    ]
    # 已依需求移除：primary_usage_scene_ratio、mean_usage_scene_ratio、
    # max_usage_scene_ratio、max_gemini_avg_score、video_count
    df_video_final = df_video[["project_id"] + video_features].copy()
    # category 欄位含「成功/失敗」字樣，已排除，不納入特徵矩陣

# ------------------------------------------------------------------
# 3. 價格競爭力（market_price_results*.csv）
# ------------------------------------------------------------------
print("[3/6] 價格特徵")
df_price = load_and_concat(FILE_PATTERNS["price"])
df_price_final = pd.DataFrame()
if not df_price.empty:
    df_price["project_id"] = df_price["project_id"].apply(extract_project_code)
    df_price["label"] = df_price["category_path"].apply(extract_label)  # 標籤來源，之後不進特徵
    cats = df_price["category_path"].apply(extract_category_platform)
    df_price["category_name"] = [c[0] for c in cats]
    df_price["platform_type"] = [c[1] for c in cats]
    df_price["market_data_quality_high"] = (df_price["market_data_quality"] == "high").astype(int)
    df_price["innovation_label_is_innovative"] = df_price["innovation_label"].str.contains("創新", na=False).astype(int)

    price_features = [
        "project_core_price", "market_price_min", "market_price_max", "price_deviation_ratio",
        "is_price_competitive", "needs_manual_review",
        "market_data_quality_high", "innovation_label_is_innovative",
    ]
    df_price_final = df_price[["project_id", "label", "platform_type", "category_name"] + price_features].copy()
    # status（API 呼叫狀態，非專案結果）/ category_path / error_message / review_reason /
    # matched_product_titles / product_keyword 皆排除

# ------------------------------------------------------------------
# 4. 影像吸引力：圖文比例（image_text_ratio_result*.csv）
# ------------------------------------------------------------------
print("[4/6] 圖文比例特徵")
df_img = load_and_concat(FILE_PATTERNS["image_text"])
df_img_final = pd.DataFrame()
if not df_img.empty:
    df_img["project_id"] = df_img["project"].apply(extract_project_code)
    df_img_final = df_img[["project_id", "word_count", "image_count", "video_count", "media_per_100_words"]].copy()
    df_img_final = df_img_final.rename(columns={
        "word_count": "img_word_count", "image_count": "img_image_count", "video_count": "img_video_count",
    })

# ------------------------------------------------------------------
# 5. training_dataset.csv —— 市場契合度 PMF 特徵
#    （trend_scores.csv 已改為讀 training_dataset.csv，只擷取 pmf_ 開頭的欄位；
#     這份檔案自帶 matched_full_project_id，可直接對到其他來源的完整代碼，
#     不需要再做短代碼比對）
# ------------------------------------------------------------------
print("[5/6] 市場契合度 PMF 特徵（training_dataset.csv）")
df_trend = load_and_concat(FILE_PATTERNS["trend"])
df_trend_final = pd.DataFrame()
trend_label_source = pd.DataFrame(columns=["project_id", "label"])
if not df_trend.empty:
    id_col = "matched_full_project_id" if "matched_full_project_id" in df_trend.columns else "project_id"
    df_trend["project_id"] = df_trend[id_col].apply(extract_project_code)

    # pmf_b_trend_level 對應附圖「Google 趨勢斜率」，要保留；
    # pmf_b_is_imputed 只是「這筆是否為補值」的旗標，不是趨勢數值，排除掉
    pmf_cols = [c for c in df_trend.columns if c.startswith("pmf_") and c != "pmf_b_is_imputed"]
    print(f"  擷取到的 PMF 欄位：{pmf_cols}")
    for c in pmf_cols:
        if df_trend[c].dtype == bool:
            df_trend[c] = df_trend[c].astype(int)  # True/False -> 1/0，XGBoost 需要數值輸入

    if "募資狀態" in df_trend.columns:
        df_trend["label"] = df_trend["募資狀態"].map({"成功": 1, "失敗": 0})
    else:
        df_trend["label"] = df_trend["project_id"].apply(extract_label)
    label_counts = df_trend["label"].value_counts(dropna=False)
    print(f"  標籤分布：{dict(label_counts)}")

    df_trend_final = df_trend[["project_id"] + pmf_cols].dropna(subset=["project_id"]).copy()
    df_trend_final = df_trend_final.drop_duplicates(subset="project_id")
    trend_label_source = df_trend[["project_id", "label"]].dropna(subset=["label"])
    # 達標率(%) / is_hit / 方案價格列表 / FAQ 相關欄位這份檔案裡也有，但那些已經由
    # projects_summary.csv 負責（或本來就該排除），這裡刻意只挑 pmf_ 開頭的欄位，
    # 不重複合併、也不會把 is_hit 這類未經確認是否洩漏的欄位帶進來。
# ------------------------------------------------------------------
# 6. projects_summary*.csv —— 專案執行力／回饋方案／FAQ 特徵
#
#    這份檔案的「達標率(%)」欄位本質上就是結果的連續版本
#    （達標率 ≥ 100 幾乎等於「成功」），刻意排除、絕不進入特徵矩陣，
#    只用「募資狀態」欄位當標籤來源（跟其他來源做法一致）。
#
#    這份檔案的「專案編號」是短代碼（例如 GS1），其他來源用的是完整代碼
#    （例如 GS1_some-slug）。兩者用「底線前綴」比對，不是完整字串相等：
#    先從已合併好的其他來源建立 短代碼→完整代碼 對照表，比對不到的短代碼
#    就直接當成新專案的代碼使用（代表這個專案還沒被其他 agent 處理過）。
# ------------------------------------------------------------------
print("[6/6] 專案執行力／回饋方案／FAQ 特徵")
df_summary = load_and_concat(FILE_PATTERNS["summary"])
df_summary_final = pd.DataFrame()
summary_label_source = pd.DataFrame(columns=["project_id", "label"])
summary_meta = pd.DataFrame()
if not df_summary.empty:
    df_summary["label"] = df_summary["募資狀態"].map({"成功": 1, "失敗": 0})  # 標籤來源，之後不進特徵
    df_summary["category_name"] = df_summary["次分類"]
    df_summary["platform_type"] = df_summary["主分類"]

    def parse_price_list(s):
        if pd.isna(s) or str(s).strip() == "":
            return []
        out = []
        for x in str(s).split("|"):
            x = x.strip()
            if x:
                try:
                    out.append(float(x))
                except ValueError:
                    pass
        return out

    price_lists = df_summary["方案價格列表"].apply(parse_price_list)
    df_summary["reward_price_min"] = price_lists.apply(lambda L: min(L) if L else None)
    df_summary["reward_price_max"] = price_lists.apply(lambda L: max(L) if L else None)
    df_summary["reward_price_mean"] = price_lists.apply(lambda L: sum(L) / len(L) if L else None)
    df_summary["reward_price_median"] = price_lists.apply(
        lambda L: sorted(L)[len(L) // 2] if L else None
    )

    # 募資天數：優先用「總天數」欄位，沒有的話從開始/結束日期算
    if "總天數" in df_summary.columns:
        df_summary["funding_duration_days"] = df_summary["總天數"]
    elif "開始日期" in df_summary.columns and "結束日期" in df_summary.columns:
        df_summary["funding_duration_days"] = (
            pd.to_datetime(df_summary["結束日期"]) - pd.to_datetime(df_summary["開始日期"])
        ).dt.days

    df_summary = df_summary.rename(columns={
        "折扣層數": "reward_tier_count",
        "FAQ總題數": "faq_total_count",
        "FAQ更新頻率": "faq_update_frequency",
    })

    # TODO 目標金額（附圖「1. 專案執行力」細項特徵，目前所有來源 CSV 都還沒有這個欄位）：
    #   1) 確認這個欄位實際在哪個 CSV、叫什麼名字（例如 projects_summary.csv 裡的
    #      「目標金額」或「募資目標」欄位，或另一份還沒接進來的檔案）
    #   2) 在這裡加一行轉換，例如：
    #        df_summary["target_amount"] = df_summary["<你的實際欄位名稱>"]
    #      （如果原始欄位是字串格式的金額，可能要先 pd.to_numeric() 或去除千分位逗號）
    #   3) 把 "target_amount" 加進下面這個清單即可，其餘不用改：
    #      resolve_feature_schema() 已經在 crowdfunding_model.py 的
    #      BASE_DIMENSION_MAP「1. 專案執行力」裡預留了 "target_amount" 這個欄位名稱，
    #      只要 merged_training_data.csv 裡真的有這一欄，就會自動被抓進特徵矩陣，
    #      不需要再改 crowdfunding_model.py。
    summary_features = [f for f in [
        "funding_duration_days", "target_amount", "reward_tier_count", "faq_total_count", "faq_update_frequency",
        "reward_price_min", "reward_price_max", "reward_price_mean", "reward_price_median",
    ] if f in df_summary.columns]

    df_summary["short_code"] = df_summary["專案編號"].astype(str).str.strip()
    df_summary_final = df_summary[["short_code", "label"] + summary_features].copy()
    summary_meta = df_summary[["short_code", "platform_type", "category_name"]].copy()
    # 達標率(%) / 專案編號 / 方案價格列表(原始字串) / 主分類 / 次分類 / 募資狀態 皆已轉換或排除，不重複進特徵矩陣


label_sources = []
if not df_copy_final.empty:
    label_sources.append(df_copy_final[["project_id", "label"]])
if not df_price_final.empty:
    label_sources.append(df_price_final[["project_id", "label"]])
if not trend_label_source.empty:
    label_sources.append(trend_label_source)

meta_sources = []
if not df_copy_final.empty:
    meta_sources.append(df_copy_final[["project_id", "platform_type", "category_name"]])
if not df_price_final.empty:
    meta_sources.append(df_price_final[["project_id", "platform_type", "category_name"]])

feature_frames = [
    df_copy_final.drop(columns=["label", "platform_type", "category_name"], errors="ignore"),
    df_video_final,
    df_price_final.drop(columns=["label", "platform_type", "category_name"], errors="ignore"),
    df_img_final,
    df_trend_final,
]
feature_frames = [f for f in feature_frames if not f.empty]

merged = feature_frames[0]
for fdf in feature_frames[1:]:
    merged = merged.merge(fdf, on="project_id", how="outer")

# --- 把 projects_summary 的短代碼（如 GS1）對應到上面已知的完整代碼（如 GS1_some-slug）---
if not df_summary_final.empty:
    full_ids = merged["project_id"].dropna().unique()
    short_to_full = {}
    ambiguous = set()
    for fid in full_ids:
        code = fid.split("_", 1)[0]
        if code in short_to_full and short_to_full[code] != fid:
            ambiguous.add(code)
        short_to_full[code] = fid
    if ambiguous:
        print(f"  ⚠️ {len(ambiguous)} 個短代碼對應到多個不同的完整代碼，"
              f"已各自取第一個匹配，建議人工核對：{sorted(ambiguous)}")

    def resolve_id(short_code):
        return short_to_full.get(short_code, short_code)  # 比對不到就當作全新專案的代碼使用

    df_summary_final = df_summary_final.copy()
    df_summary_final["project_id"] = df_summary_final["short_code"].apply(resolve_id)
    df_summary_final = df_summary_final.drop(columns=["short_code"])
    n_new = (~df_summary_final["project_id"].isin(full_ids)).sum()
    if n_new:
        print(f"  {n_new} 筆專案只出現在 projects_summary，其他來源還沒有這些專案的資料"
              f"（會以新專案併入，缺少的其他維度欄位留 NaN）。")

    summary_meta["project_id"] = summary_meta["short_code"].apply(resolve_id)
    summary_meta = summary_meta.drop(columns=["short_code"])
    meta_sources.append(summary_meta[["project_id", "platform_type", "category_name"]])

    label_sources.append(df_summary_final[["project_id", "label"]].dropna(subset=["label"]))

    merged = merged.merge(df_summary_final.drop(columns=["label"]), on="project_id", how="outer")

labels = pd.concat(label_sources).drop_duplicates(subset="project_id").set_index("project_id")["label"]
meta = pd.concat(meta_sources).drop_duplicates(subset="project_id").set_index("project_id") if meta_sources else pd.DataFrame()

merged["label"] = merged["project_id"].map(labels)
if not meta.empty:
    merged = merged.merge(meta, on="project_id", how="left")

# 只保留有標籤的專案（無法確認結果的資料不能拿來訓練）
before = len(merged)
merged = merged.dropna(subset=["label"]).reset_index(drop=True)
merged["label"] = merged["label"].astype(int)

# category_name / platform_type 目前是文字欄位，XGBoost 需要數值輸入，
# 在這裡展開成 one-hot（欄名會變成 category_時尚 / platform_群眾募資 這種格式，
# 剛好符合 crowdfunding_model.py 自動偵測「4. 專案屬性」用的 category_ / platform_ 前綴規則）。
# 新增類別時這裡會自動多出新的欄位，不需要改程式碼。
dummy_cols = [c for c in ("category_name", "platform_type") if c in merged.columns]
if dummy_cols:
    merged = pd.get_dummies(
        merged, columns=dummy_cols,
        prefix=["category" if c == "category_name" else "platform" for c in dummy_cols],
        dummy_na=False,
    )
    merged[[c for c in merged.columns if c.startswith(("category_", "platform_"))]] = \
        merged[[c for c in merged.columns if c.startswith(("category_", "platform_"))]].astype(int)

print(f"\n合併後總專案數：{before}，其中有效標籤（可用於訓練）：{len(merged)}")
print(f"標籤分布：\n{merged['label'].value_counts()}")
cat_cols = [c for c in merged.columns if c.startswith("category_")]
if cat_cols:
    print("類別分布：")
    for c in cat_cols:
        print(f"  {c}: {int(merged[c].sum())}")
print(f"\n特徵欄位共 {merged.shape[1] - 2} 個（不含 project_id / label）")
missing_rate = merged.drop(columns=["project_id", "label"]).isna().mean().sort_values(ascending=False)
print("\n各特徵缺值比例（前10高）：")
print(missing_rate.head(10))

merged.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n已輸出：{OUT_PATH}")


## 6. `crowdfunding_model.py` —— 訓練、校準驗證、SHAP 解釋

- 訓練集／校準集／測試集三分法，Isotonic Regression 只在校準集上擬合，效果在完全獨立的測試集上重新驗證
- 特徵維度是動態偵測的：CSV 裡有哪些欄位就用哪些，不假設固定的類別數量
- 訓練前會先跑**洩漏偵測**：對每個特徵單獨算 AUC（單一特徵 AUC 接近 1 很可疑），
  並檢查每個類別/平台 one-hot 欄位是不是「樣本太少又剛好結果太純」
- 除了原本的 SHAP 分佈圖，另外輸出「特徵相關係數熱力圖」跟「特徵影響程度排行長條圖」

In [ ]:
# -*- coding: utf-8 -*-
"""
crowdfunding_model.py
======================
募資專案成功機率預測模型（XGBoost + SHAP + 機率校準驗證）

功能：
  1. 讀取使用者收集的 CSV（16 特徵 / 5 維度）
  2. 以「訓練集 / 校準集 / 測試集」三分法訓練 XGBoost 分類器
  3. 在校準集上計算 Brier Score 與可靠度曲線（Calibration Curve），
     診斷模型是否過度自信（overconfident）或保守（underconfident）
  4. 若偏移超過門檻，自動導入保序回歸（Isotonic Regression）進行後處理校準，
     並在「未參與校準的測試集」上重新驗證 Brier Score，證明校準確實改善品質
  5. 用 SHAP（TreeExplainer）計算每個特徵對單一專案成功機率的貢獻比例
  6. 輸出：成功機率（校準後）＋ 各特徵 / 各維度貢獻百分比 ＋ 完整驗證報告

使用方式：
  python crowdfunding_model.py --csv your_data.csv --target 是否成功
  （若不指定 --csv，會用內建模擬資料跑一次完整流程做為示範/單元測試）
"""

import argparse
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from scipy.special import expit
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    brier_score_loss, roc_auc_score, log_loss, average_precision_score,
    f1_score, precision_score, recall_score, confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# ==========================================================
# 特徵影響力權重（不刪除特徵，只降低它在建樹時被選中的機率）
# ==========================================================
# 這是 XGBoost 原生支援的做法：訓練時每次要在某個節點選特徵來分裂，
# 會依 feature_weights 的比例做加權抽樣（要搭配 colsample_bytree/bynode < 1
# 才有效果，本檔案的 train_xgb() 已經有設 colsample_bytree=0.85）。
# 權重 1.0 = 正常；越接近 0 = 越少被選中分裂、影響力越低；設 0 幾乎等於排除，
# 但仍保留在特徵矩陣裡（跟直接從 BASE_DIMENSION_MAP 刪掉不同，那是徹底移除）。
# 沒列在這裡的特徵一律用 DEFAULT_FEATURE_WEIGHT。
FEATURE_INFLUENCE_WEIGHTS: dict[str, float] = {
    # 範例：懷疑 img_image_count 可能有洩漏風險、或影響力過大，先降權觀察，
    # 而不是直接刪除（刪除是 BASE_DIMENSION_MAP 那邊的做法）：
    # "img_image_count": 0.3,
    # "feat_trust_score": 0.5,
}
DEFAULT_FEATURE_WEIGHT = 1.0

# ==========================================================
# 0. 特徵維度對應表（基礎版）—— 對應 build_dataset.py 合併出來的欄位
#    維度命名比照使用者提供的特徵維度分類表（專案執行力／價格競爭力／
#    文案說服力／影像吸引力／市場契合度），但不侷限於表上列出的細項——
#    只要是同一種性質的特徵，都歸進對應維度；「6. 專案屬性」（類別 one-hot、
#    平台類型等）是動態的，CSV 有多少類別就自動長出多少欄位，不假設
#    只有固定幾種——之後加入更多類別時，這裡不需要手動改。
# ==========================================================
BASE_DIMENSION_MAP = {
    "1. 專案執行力": [
        "funding_duration_days",
        "target_amount",  # 目標金額（附圖細項特徵）——先預留欄位名稱，等 build_dataset.py
                           # 接上實際來源欄位後，merged_training_data.csv 裡有這一欄就會自動被抓進來
    ],
    "2. 價格競爭力": [
        "project_core_price", "market_price_min", "market_price_max", "price_deviation_ratio",
        "is_price_competitive", "needs_manual_review",
        "market_data_quality_high", "innovation_label_is_innovative",
        "reward_tier_count", "reward_price_min", "reward_price_max",
        "reward_price_mean", "reward_price_median",
    ],
    "3. 文案說服力": [
        "feat_text_story_ratio", "feat_text_spec_ratio_rule", "feat_text_risk_ratio",
        "feat_has_social_link", "feat_punct_intensity", "feat_avg_sentence_len",
        "feat_type_token_ratio", "feat_trust_score", "feat_emotional_ratio", "feat_spec_ratio",
        "feat_emotional_word_count", "feat_spec_word_count",
        "faq_total_count", "faq_update_frequency",
        "img_word_count",  # 對應附圖「文案字數」，依附圖歸類調整，從「4. 影像吸引力」移過來
    ],
    "4. 影像吸引力": [
        "videos_included", "videos_excluded_low_coverage", "total_video_duration",
        "primary_usage_scene_soft_ratio", "primary_gemini_avg_score",
        "mean_usage_scene_soft_ratio",
        "max_usage_scene_soft_ratio", "mean_gemini_avg_score",
        "duration_weighted_usage_scene_ratio",
        "img_image_count", "img_video_count", "media_per_100_words",
    ],
    # 已依需求移除：video_count、primary_usage_scene_ratio、mean_usage_scene_ratio、
    # max_usage_scene_ratio、max_gemini_avg_score
    # 已依附圖歸類調整：img_word_count（文案字數）改歸入「3. 文案說服力」
    "5. 市場契合度(PMF)": [
        "pmf_a_recent_category_success_rate", "pmf_b_trend_level",  # pmf_b_trend_level = Google 趨勢斜率
        "pmf_c_hit_similarity", "pmf_d_market_saturation_count",
    ],
}
BASE_FEATURES = [f for feats in BASE_DIMENSION_MAP.values() for f in feats]

# 動態偵測用的前綴：CSV 裡任何欄位符合這些規則，都會自動歸入「6. 專案屬性」
DYNAMIC_ATTR_PREFIXES = ("category_", "platform_")


def resolve_feature_schema(df: pd.DataFrame):
    """
    根據實際拿到的 CSV 欄位，組出這次要用的完整特徵清單與維度對應表。
    已知的核心特徵（BASE_DIMENSION_MAP）只取 CSV 裡實際存在的部分；
    任何符合 category_ / platform_ 前綴的欄位（例如未來新增的類別 one-hot）
    都自動歸類到「6. 專案屬性」，不需要事先知道會有幾種類別。
    """
    dimension_map = {}
    for dim, feats in BASE_DIMENSION_MAP.items():
        present = [f for f in feats if f in df.columns]
        if present:
            dimension_map[dim] = present

    dynamic_attrs = sorted(
        c for c in df.columns
        if c.startswith(DYNAMIC_ATTR_PREFIXES) and c not in BASE_FEATURES
    )
    if dynamic_attrs:
        dimension_map["6. 專案屬性"] = dynamic_attrs

    all_features = [f for feats in dimension_map.values() for f in feats]
    return all_features, dimension_map


MODEL_BUNDLE_VERSION = 1


def save_model_bundle(path, model, iso, all_features, dimension_map, target_col):
    """
    把訓練完的模型打包成一個檔案，之後不用重新訓練就能直接拿來預測新專案。
    打包內容：XGBoost 模型本身、Isotonic 校準器（可能是 None，代表這次沒套用校準）、
    這次訓練實際用到的特徵清單與維度對應表（新專案的欄位要對齊這份清單，
    多的欄位會被忽略、缺的欄位會補 NaN，交給 XGBoost 原生處理）。
    """
    bundle = {
        "version": MODEL_BUNDLE_VERSION,
        "model": model,
        "isotonic": iso,
        "all_features": all_features,
        "dimension_map": dimension_map,
        "target_col": target_col,
    }
    joblib.dump(bundle, path)


def load_model_bundle(path):
    bundle = joblib.load(path)
    if bundle.get("version") != MODEL_BUNDLE_VERSION:
        print(f"⚠️ 模型檔版本（{bundle.get('version')}）跟目前程式版本（{MODEL_BUNDLE_VERSION}）不同，"
              f"欄位對應方式可能有差異，建議重新訓練後再產生一次模型檔。")
    return bundle


# ==========================================================
# 1. 資料載入 / 內建模擬資料（無 CSV 時的示範與自我測試用）
# ==========================================================
def load_csv_data(csv_path: str, target_col: str):
    """讀取使用者 CSV。不要求所有已知特徵都存在——有多少特徵用多少，
    未來新增類別、新增特徵欄位都不需要改這支程式。"""
    df = pd.read_csv(csv_path)
    if target_col not in df.columns:
        raise ValueError(
            f"CSV 缺少目標欄位 `{target_col}`（1=成功, 0=失敗）。"
            f"目前欄位有：{list(df.columns)}"
        )
    all_features, dimension_map = resolve_feature_schema(df)
    if not all_features:
        raise ValueError("CSV 裡找不到任何已知的特徵欄位，請確認欄位名稱是否一致。")
    df = df[all_features + [target_col]].dropna(subset=[target_col])
    # 注意：這裡刻意不補值。跨類別/跨特徵來源互缺的欄位，NaN 是
    # 「這個維度沒有蒐集到資料」的結構性缺失，補中位數反而會抹掉這個訊號；
    # XGBoost 原生就能處理 NaN（分裂時自動學習缺值該往哪個子節點走）。
    return df, all_features, dimension_map


def make_synthetic_data(n_samples: int = 1500):
    """僅供沒有真實 CSV 時的示範／流程自我測試，不用於實際預測。
    模擬「三個類別」（刻意不是兩個）且各類別互缺欄位的真實結構，
    用來驗證動態特徵偵測與 NaN 處理在類別數量變動時仍能正確運作。"""
    rng = np.random.RandomState(RANDOM_STATE)
    categories = ["時尚", "教育", "3C"]
    cat_choice = rng.choice(categories, n_samples)

    df = pd.DataFrame({f: rng.uniform(0, 1, n_samples) for f in BASE_FEATURES})
    # 時尚：只有文案+影片特徵；教育：只有價格特徵；3C：文案+價格都有（模擬更完整的未來資料）
    df.loc[cat_choice == "教育", BASE_DIMENSION_MAP["3. 文案說服力"]] = np.nan
    df.loc[cat_choice != "3C", []] = df.loc[cat_choice != "3C", []]  # no-op，保留結構清晰
    df.loc[cat_choice == "時尚", BASE_DIMENSION_MAP["2. 價格競爭力"]] = np.nan
    df.loc[cat_choice == "教育", ["img_word_count", "img_image_count", "img_video_count", "media_per_100_words"]] = np.nan

    for c in categories:
        df[f"category_{c}"] = (cat_choice == c).astype(int)
    df["platform_is_preorder"] = rng.randint(0, 2, n_samples)

    logit = (
        df["feat_trust_score"].fillna(0) / 50
        + df["is_price_competitive"].fillna(0) * 1.2
        - df["price_deviation_ratio"].fillna(0) * 0.5
        + rng.normal(0, 0.6, n_samples)
    )
    df["label"] = (logit > np.nanmedian(logit)).astype(int)
    return df


# ==========================================================
# 1.5 洩漏偵測 —— 訓練前先檢查有沒有「太好到不像真的」的特徵
# ==========================================================
def univariate_leakage_scan(X: pd.DataFrame, y: pd.Series, auc_flag_threshold: float = 0.95):
    """
    對每一個特徵『單獨』算一次 AUC（不看其他特徵、不用模型，只看這一欄自己
    能把成功/失敗分得多開）。真正有意義的特徵通常單獨看不會分得太乾淨
    （現實世界的訊號都是有雜訊的）；如果某一欄自己就能算出 AUC 接近 1，
    很可能不是「這個特徵超強」，而是它在製作過程中偷看到了答案。

    回傳一個依 AUC 排序的表格，AUC 越接近 1 越可疑（1 代表完美分開，
    0.5 代表跟亂猜一樣沒有分辨力）。
    """
    rows = []
    for col in X.columns:
        s = X[col]
        mask = s.notna()
        if mask.sum() < 10 or y[mask].nunique() < 2:
            continue  # 樣本太少或該欄有值的資料裡只有單一類別，算不出有意義的 AUC
        try:
            auc = roc_auc_score(y[mask], s[mask])
        except (ValueError, TypeError):
            continue
        auc = max(auc, 1 - auc)  # 特徵可能跟結果正相關或負相關，取「分辨力」不看方向
        rows.append({"feature": col, "univariate_auc": round(auc, 4), "n_available": int(mask.sum())})

    report = pd.DataFrame(rows).sort_values("univariate_auc", ascending=False).reset_index(drop=True)
    suspicious = report[report["univariate_auc"] >= auc_flag_threshold]
    return report, suspicious


def check_category_purity(df: pd.DataFrame, target_col: str, dimension_map: dict, min_n: int = 30):
    """
    專門檢查「4. 專案屬性」裡的類別/平台 one-hot 欄位：
    如果某個類別的樣本數很少、又剛好全部同一種結果（100% 成功或 100% 失敗），
    模型可能只是學到「看到這個類別就直接猜對應結果」，而不是真的學到有用的訊號
    ——這在類別數量增加、但每個類別樣本數還不夠多時特別容易發生。
    """
    attr_cols = dimension_map.get("6. 專案屬性", [])
    rows = []
    for col in attr_cols:
        if col not in df.columns:
            continue
        sub = df[df[col] == 1]
        n = len(sub)
        if n == 0:
            continue
        rate = sub[target_col].mean()
        risky = bool(n < min_n and rate in (0.0, 1.0))
        rows.append({"attribute": col, "n": n, "success_rate": round(float(rate), 3), "risky_pure": risky})
    return pd.DataFrame(rows).sort_values("n").reset_index(drop=True)


def print_leakage_report(X: pd.DataFrame, y: pd.Series, dimension_map: dict, df_with_label: pd.DataFrame, target_col: str):
    print("\n[洩漏 / 類別純度檢查]")
    report, suspicious = univariate_leakage_scan(X, y)
    print("  單一特徵 AUC 排行（前 8 名，越接近 1 越可疑）：")
    for _, row in report.head(8).iterrows():
        flag = " ⚠️ 疑似洩漏" if row["univariate_auc"] >= 0.95 else ""
        print(f"    {row['feature']:<30} AUC={row['univariate_auc']:.4f}（可用樣本 {row['n_available']}）{flag}")
    if suspicious.empty:
        print("  ✅ 沒有單一特徵的 AUC 超過 0.95，暫無明顯洩漏跡象。")
    else:
        print(f"  ⚠️ 有 {len(suspicious)} 個特徵單獨 AUC ≥ 0.95，建議人工檢查這些欄位的計算方式：")
        print(f"     {list(suspicious['feature'])}")

    purity = check_category_purity(df_with_label, target_col, dimension_map)
    if not purity.empty:
        print("\n  類別/平台純度檢查：")
        for _, row in purity.iterrows():
            flag = " ⚠️ 樣本少且結果太純，建議留意" if row["risky_pure"] else ""
            print(f"    {row['attribute']:<25} n={row['n']:<5} 成功率={row['success_rate']:.3f}{flag}")
    return report, purity


# ==========================================================
# 2. 訓練 / 校準診斷 / 保序回歸校準
# ==========================================================
def train_xgb(X_train, y_train, X_val, y_val):
    """訓練 XGBoost，以獨立驗證集做 early stopping，避免過擬合。
    若 FEATURE_INFLUENCE_WEIGHTS 有設定，會依欄位順序組成權重陣列，
    降低指定特徵在建樹分裂時被選中的機率（需搭配 colsample_bytree<1 才有效果，
    這裡已經是 0.85）。"""
    model = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
    )
    feature_weights = None
    if FEATURE_INFLUENCE_WEIGHTS:
        feature_weights = np.array([
            FEATURE_INFLUENCE_WEIGHTS.get(col, DEFAULT_FEATURE_WEIGHT)
            for col in X_train.columns
        ], dtype=float)
        applied = {k: v for k, v in FEATURE_INFLUENCE_WEIGHTS.items() if k in X_train.columns}
        if applied:
            print(f"[特徵影響力權重] 已套用降權：{applied}")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
        feature_weights=feature_weights,
    )
    return model


def brier_skill_score(y_true, y_prob) -> float:
    """
    Brier Skill Score：把模型的 Brier Score 拿去跟「什麼都不看、永遠猜整體平均成功率」
    這個最笨的基準比較，換算成一個 0~1 的「贏過笨基準多少」分數（1=完美，0=跟瞎猜一樣，
    負數=比瞎猜還差）。原始 Brier Score 的高低會受基礎成功率影響，換一批基礎成功率不同
    的資料，同一個模型的 Brier 會跟著變動；Skill Score 排除了這個干擾，比較適合拿來
    對照不同資料集或不同時間點訓練出來的模型。
    """
    baseline_rate = float(np.mean(y_true))
    baseline_prob = np.full_like(np.asarray(y_prob, dtype=float), baseline_rate)
    baseline_brier = brier_score_loss(y_true, baseline_prob)
    model_brier = brier_score_loss(y_true, y_prob)
    if baseline_brier == 0:
        return float("nan")
    return 1 - model_brier / baseline_brier


def score_to_grade(score_100: float) -> str:
    """
    把 0~100 分的成功機率分數，對應成一般人比較好理解的等第。
    分數本身就是「校準後成功機率 x 100」，等第只是方便閱讀的分箱，
    門檻可以依你們實際的成功率分布再調整，這裡先給一個常見的預設切法。
    """
    if score_100 >= 90:
        return "A+（強烈看好）"
    elif score_100 >= 80:
        return "A（表現優異）"
    elif score_100 >= 70:
        return "B+（穩健）"
    elif score_100 >= 60:
        return "B（普通偏正向）"
    elif score_100 >= 50:
        return "C（偏保留，風險偏高）"
    else:
        return "D（高風險，不建議樂觀預期）"


def compute_quality_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:
    """
    除了既有的 AUC / LogLoss / Brier / ECE 之外，再補幾個檢查模型品質的常用指標：
      - PR-AUC（Average Precision）：label 不平衡時，比 ROC-AUC 更能反映「抓到真正
        會成功的專案」的能力，不會被大量的「輕鬆判斷」樣本撐高分數。
      - F1 / Precision / Recall（@ threshold）：把機率換算成「預測成功 / 預測失敗」的
        二元決策後，實際抓對、抓錯的比例——比單看機率更貼近「這個模型能不能拿來做
        篩選/決策」的問題。
      - 混淆矩陣（TN/FP/FN/TP）：Precision/Recall 背後的原始數字，方便你自己抓別的
        threshold 重算，或跟業務端討論「寧可錯殺還是寧可放過」的取捨。
      - KS 統計量（Kolmogorov-Smirnov）：信用評分/募資評分類模型很常用的指標，
        衡量「成功組」與「失敗組」的預測機率分布，最多能被模型拉開多遠（0~1，
        越高代表模型越能把兩群完全分開，業界常見門檻 KS>=0.3 算堪用、>=0.5 算不錯）。
      - 前 10% 高分組 Lift：把測試集依預測機率由高到低排序，取前 10%，看這群人的
        實際成功率是整體平均成功率的幾倍——如果 Lift 接近 1，代表模型排序前段的
        專案跟隨機挑沒兩樣，是很直觀的「模型到底有沒有用」的檢查。
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    pr_auc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # KS 統計量：成功組 / 失敗組的機率分布，累積分布函數最大差距
    order = np.argsort(y_prob)
    y_true_sorted = y_true[order]
    n_pos = y_true_sorted.sum()
    n_neg = len(y_true_sorted) - n_pos
    if n_pos > 0 and n_neg > 0:
        cum_pos = np.cumsum(y_true_sorted) / n_pos
        cum_neg = np.cumsum(1 - y_true_sorted) / n_neg
        ks = float(np.max(np.abs(cum_pos - cum_neg)))
    else:
        ks = float("nan")

    # 前 10% 高分組 Lift
    top_n = max(1, int(len(y_prob) * 0.1))
    top_idx = np.argsort(y_prob)[::-1][:top_n]
    top_decile_success_rate = float(y_true[top_idx].mean())
    overall_success_rate = float(y_true.mean())
    lift = top_decile_success_rate / overall_success_rate if overall_success_rate > 0 else float("nan")

    return {
        "pr_auc": float(pr_auc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "ks_statistic": ks,
        "top_decile_success_rate": top_decile_success_rate,
        "overall_success_rate": overall_success_rate,
        "top_decile_lift": float(lift),
    }


def per_segment_discrimination(y_test, test_prob_raw, X_test, dimension_map, min_n: int = 30):
    """
    分類別／分平台拆開算 AUC 與 Brier Score。

    整體 AUC 可能被「某個類別基礎成功率特別高」撐高，掩蓋掉模型在單一類別內部其實
    分不太出來的事實（例如 category_科技 成功率 90%，模型只要學會「看到科技類就猜
    成功」，不用真的看懂文案或影片內容，整體 AUC 照樣很漂亮）。這裡针對「6. 專案屬性」
    裡的每個 one-hot 類別/平台欄位，只挑出該欄位=1 的子集合，各自重新算一次 AUC 跟
    Brier Score，藉此檢查模型是不是在「每個類別內部」都還有辨別力，而不只是在吃
    類別間的基礎成功率差異。
    """
    attr_cols = dimension_map.get("6. 專案屬性", [])
    results = []
    for col in attr_cols:
        if col not in X_test.columns:
            continue
        mask = X_test[col].fillna(0).astype(int) == 1
        n = int(mask.sum())
        if n < min_n:
            continue
        y_sub = y_test[mask.values]
        if y_sub.nunique() < 2:
            continue
        prob_sub = test_prob_raw[mask.values]
        results.append({
            "segment": col,
            "n": n,
            "auc": roc_auc_score(y_sub, prob_sub),
            "brier": brier_score_loss(y_sub, prob_sub),
        })
    return results


def expected_calibration_error(y_true, y_prob, n_bins: int = 10) -> float:
    """ECE：各機率區間內「預測平均機率」與「實際成功率」差距的加權平均。"""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins[1:-1])
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_conf = y_prob[mask].mean()
        bin_acc = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(bin_conf - bin_acc)
    return ece


def diagnose_calibration(y_true, y_prob, label: str):
    """計算 Brier Score + 可靠度曲線 + ECE，回傳診斷結果字典。"""
    brier = brier_score_loss(y_true, y_prob)
    ece = expected_calibration_error(y_true, y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    mean_bias = np.mean(prob_pred - prob_true)  # >0 代表過度自信；<0 代表保守
    return {
        "brier_score": brier,
        "ece": ece,
        "mean_bias": mean_bias,
        "prob_true": prob_true,
        "prob_pred": prob_pred,
        "label": label,
    }


def _force_cjk_font_on_all_text(fig, font_candidates=("WenQuanYi Zen Hei", "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans CJK KR")):
    """
    shap.summary_plot 對 y 軸特徵名稱標籤有自己的一套文字設定方式，
    不完全遵守全域 matplotlib.rcParams 的字型設定，導致中文（尤其是
    category_/platform_ 這類動態 one-hot 出來的中文類別名稱）顯示成方框。
    這裡在畫完圖之後，強制把整張圖上每一個文字物件的字型都改成可用的
    CJK 字型，確保不管 SHAP 內部怎麼設定，最後存檔的圖都能正確顯示中文。
    """
    available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
    font_name = next((f for f in font_candidates if f in available), None)
    if font_name is None:
        return  # 找不到任何候選字型就放棄，不強制設定，避免報錯
    for ax in fig.get_axes():
        texts = ax.get_xticklabels() + ax.get_yticklabels()
        texts += [ax.xaxis.label, ax.yaxis.label, ax.title]
        if ax.get_legend() is not None:
            texts += ax.get_legend().get_texts()
        for t in texts:
            t.set_fontfamily(font_name)


def plot_calibration(diag_before: dict, diag_after: dict | None, out_path: str):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="完美校準")
    ax.plot(diag_before["prob_pred"], diag_before["prob_true"], marker="o",
             label=f"校準前 (Brier={diag_before['brier_score']:.4f})")
    if diag_after is not None:
        ax.plot(diag_after["prob_pred"], diag_after["prob_true"], marker="s",
                 label=f"校準後-Isotonic (Brier={diag_after['brier_score']:.4f})")
    ax.set_xlabel("模型預測機率（分箱平均）")
    ax.set_ylabel("實際歷史成功率（分箱平均）")
    ax.set_title("可靠度曲線 (Calibration Curve)")
    ax.legend()
    fig.tight_layout()
    _force_cjk_font_on_all_text(fig)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def calibrate_if_needed(model, X_cal, y_cal, X_test, y_test,
                         ece_threshold: float = 0.05):
    """
    在校準集上診斷模型：
      - 若 ECE 超過門檻，代表存在顯著過度自信/保守偏移
      - 用校準集（未曾用於訓練/early-stopping）擬合 Isotonic Regression
      - 在完全獨立的測試集上重新驗證 Brier Score，證明校準確有效果
    回傳：(是否套用校準, 測試集上校準前診斷, 測試集上校準後診斷(或 None), isotonic 物件(或 None))
    """
    raw_cal_prob = model.predict_proba(X_cal)[:, 1]
    diag_cal = diagnose_calibration(y_cal, raw_cal_prob, label="校準集-校準前")

    raw_test_prob = model.predict_proba(X_test)[:, 1]
    diag_test_before = diagnose_calibration(y_test, raw_test_prob, label="測試集-校準前")

    needs_calibration = diag_cal["ece"] > ece_threshold
    if not needs_calibration:
        return False, diag_test_before, None, None

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_cal_prob, y_cal)

    calibrated_test_prob = iso.transform(raw_test_prob)
    diag_test_after = diagnose_calibration(y_test, calibrated_test_prob, label="測試集-校準後")

    return True, diag_test_before, diag_test_after, iso


# ==========================================================
# 3. SHAP 特徵貢獻（單一專案解釋 + 全域重要性）
# ==========================================================
def build_shap_explainer(model, X_background):
    # tree_path_dependent 不需背景資料，且相容於 XGBoost 的 categorical split 設定
    return shap.TreeExplainer(model, feature_perturbation="tree_path_dependent", model_output="raw")


def plot_feature_correlation_heatmap(X: pd.DataFrame, out_path: str, max_features: int | None = None):
    """
    特徵彼此之間的相關係數熱力圖。用途：
      - 找出高度共線的特徵組（例如兩個欄位幾乎是同一件事的不同寫法）
      - 也可以順便看哪些特徵跟其他特徵長得特別不一樣（可能是雜訊或需要再檢查）
    依需求一律顯示「全部特徵」，不做前 N 名截斷；max_features 保留參數但預設 None（不截斷），
    圖會依特徵數量自動放大尺寸，避免特徵一多就擠在一起看不清楚。
    """
    corr = X.corr(numeric_only=True)
    if max_features is not None and corr.shape[0] > max_features:
        avg_abs_corr = corr.abs().mean().sort_values(ascending=False)
        keep = avg_abs_corr.head(max_features).index
        corr = corr.loc[keep, keep]

    fig_size = max(6, corr.shape[0] * 0.35)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(corr.columns, fontsize=7)
    ax.set_title("特徵相關係數熱力圖 (Feature Correlation Heatmap)")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("相關係數")
    fig.tight_layout()
    _force_cjk_font_on_all_text(fig)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_shap_importance_bar(shap_values, feature_names, out_path: str, top_n: int | None = None):
    """
    每個特徵的『平均影響程度』長條圖：對每個特徵取 |SHAP value| 的平均，
    數字越大代表這個特徵平均而言把預測機率推得越遠（不分正負方向）。
    跟 shap_summary.png（看每個特徵怎麼影響、正負方向、跟特徵值高低的關係）是互補的兩張圖：
    這張回答「哪個特徵整體最重要」，summary 圖回答「這個特徵是怎麼影響的」。
    依需求一律顯示「全部特徵」，top_n 保留參數但預設 None（不截斷）。
    """
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    n_show = top_n if top_n is not None else len(feature_names)
    order = np.argsort(mean_abs_shap)[::-1][:n_show]
    names = [feature_names[i] for i in order][::-1]
    values = [mean_abs_shap[i] for i in order][::-1]

    fig, ax = plt.subplots(figsize=(8, max(4, len(names) * 0.3)))
    ax.barh(names, values, color="#4C72B0")
    ax.set_xlabel("平均 |SHAP value|（影響程度）")
    ax.set_title("特徵影響程度排行 (Mean |SHAP value|)")
    fig.tight_layout()
    _force_cjk_font_on_all_text(fig)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def explain_project(explainer, model, iso, x_row: pd.DataFrame, all_features: list, dimension_map: dict):
    """
    對單一專案計算：
      - 原始機率 / 校準後最終機率
      - 每個特徵的 SHAP 貢獻，換算為「對成功機率變動的貢獻百分比」
      - 各維度加總後的健康度分數
    """
    shap_values = explainer.shap_values(x_row)       # log-odds 空間
    base_log_odds = explainer.expected_value
    if isinstance(base_log_odds, (list, np.ndarray)):
        base_log_odds = base_log_odds[0]

    feat_shap = shap_values[0]
    total_log_odds = base_log_odds + feat_shap.sum()

    base_prob = float(expit(base_log_odds))
    raw_prob = float(expit(total_log_odds))
    final_prob = float(iso.transform([raw_prob])[0]) if iso is not None else raw_prob

    # 依「原始機率 - 基準機率」的總變化量，依 SHAP 比例分配到各特徵
    total_prob_shift = raw_prob - base_prob
    sum_abs_shap = np.sum(np.abs(feat_shap))
    if sum_abs_shap > 0:
        feature_contrib_pct = (feat_shap / sum_abs_shap) * 100
    else:
        feature_contrib_pct = np.zeros_like(feat_shap)

    feature_report = sorted(
        zip(all_features, feature_contrib_pct, feat_shap),
        key=lambda t: abs(t[1]), reverse=True,
    )

    dimension_report = {}
    for dim, feats in dimension_map.items():
        idxs = [all_features.index(f) for f in feats]
        dim_pct = feature_contrib_pct[idxs].sum()
        dimension_report[dim] = round(float(dim_pct), 2)

    score_100 = round(final_prob * 100, 1)
    return {
        "base_market_probability": round(base_prob * 100, 1),
        "raw_model_probability": round(raw_prob * 100, 1),
        "final_calibrated_probability": round(final_prob * 100, 1),
        "score_100": score_100,          # 跟 final_calibrated_probability 數值相同，
                                          # 只是換個「滿分 100 分」的說法方便非技術人員閱讀
        "grade": score_to_grade(score_100),
        "total_probability_shift_pct_pts": round(total_prob_shift * 100, 1),
        "feature_contribution_pct": {f: round(float(p), 2) for f, p, _ in feature_report},
        "dimension_contribution_pct": dimension_report,
    }


# ==========================================================
# 4. 主流程
# ==========================================================
def run_pipeline(csv_path: str | None, target_col: str, output_dir: str,
                  project_index: int = 0, ece_threshold: float = 0.05):
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if csv_path:
        df, all_features, dimension_map = load_csv_data(csv_path, target_col)
        data_source = f"使用者提供 CSV：{csv_path}"
    else:
        df = make_synthetic_data()
        target_col = "label"
        all_features, dimension_map = resolve_feature_schema(df)
        data_source = "⚠️ 未提供 CSV，使用內建模擬資料（僅供流程自我測試，非真實預測）"

    print(f"[偵測到的特徵維度] {list(dimension_map.keys())}")
    for dim, feats in dimension_map.items():
        print(f"   {dim}：{len(feats)} 個特徵")

    X = df[all_features]
    y = df[target_col].astype(int)

    print_leakage_report(X, y, dimension_map, df, target_col)

    # 三分法：60% 訓練 / 20% 校準 / 20% 測試，皆用 stratify 保持成功率一致
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
    X_cal, X_test, y_cal, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)
    # 再從訓練集切一小份作為 early-stopping 驗證集（不動用校準集/測試集）
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE)

    print(f"[資料來源] {data_source}")
    print(f"[資料切分] 訓練={len(X_tr)}　早停驗證={len(X_val)}　校準集={len(X_cal)}　測試集={len(X_test)}\n")

    model = train_xgb(X_tr, y_tr, X_val, y_val)

    test_prob_raw = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, test_prob_raw)
    ll = log_loss(y_test, test_prob_raw)
    print(f"[判別力指標] 測試集 AUC={auc:.4f}　LogLoss={ll:.4f}")

    # 分類別／分平台檢查模型是不是只在吃「類別間基礎成功率差異」，
    # 而不是在每個類別「內部」也看得出誰會成功、誰會失敗
    segment_results = per_segment_discrimination(y_test, test_prob_raw, X_test, dimension_map)
    if segment_results:
        print("\n[分類別/分平台判別力檢查]（整體 AUC 高，不代表每個子群內部都分得出來）")
        for r in sorted(segment_results, key=lambda x: x["auc"]):
            flag = "⚠️ 低於整體 AUC 很多，可能在吃類別基礎成功率" if r["auc"] < auc - 0.15 else ""
            print(f"  {r['segment']:30s} n={r['n']:4d}  AUC={r['auc']:.4f}  Brier={r['brier']:.4f}  {flag}")

    # ---- 額外模型品質指標（PR-AUC / F1 / Precision / Recall / 混淆矩陣 / KS / Lift）----
    qm = compute_quality_metrics(y_test.values, test_prob_raw, threshold=0.5)
    print("\n[模型品質補充指標]（門檻 0.5：機率 >= 0.5 視為預測成功）")
    print(f"  PR-AUC (Average Precision) = {qm['pr_auc']:.4f}")
    print(f"  Precision={qm['precision']:.4f}　Recall={qm['recall']:.4f}　F1={qm['f1']:.4f}")
    cm = qm["confusion_matrix"]
    print(f"  混淆矩陣｜TP={cm['tp']}　FP={cm['fp']}　TN={cm['tn']}　FN={cm['fn']}")
    print(f"  KS 統計量 = {qm['ks_statistic']:.4f}"
          f"（經驗法則：>=0.3 堪用，>=0.5 不錯，此值僅供參考，非絕對標準）")
    print(f"  前 10% 高分組成功率 = {qm['top_decile_success_rate']:.1%}　"
          f"整體成功率 = {qm['overall_success_rate']:.1%}　"
          f"Lift = {qm['top_decile_lift']:.2f}x"
          f"（Lift 接近 1 代表模型排序前段的專案跟隨機挑沒兩樣）")

    applied, diag_before, diag_after, iso = calibrate_if_needed(
        model, X_cal, y_cal, X_test, y_test, ece_threshold=ece_threshold)

    print("\n[機率校準驗證報告]")
    print(f"  校準前｜Brier Score={diag_before['brier_score']:.4f}　"
          f"ECE={expected_calibration_error(y_test.values, test_prob_raw):.4f}　"
          f"平均偏移={np.mean(diag_before['prob_pred'] - diag_before['prob_true']):+.4f} "
          f"({'過度自信' if np.mean(diag_before['prob_pred'] - diag_before['prob_true']) > 0 else '保守'})")

    if applied:
        print(f"  → 校準集 ECE 超過門檻 {ece_threshold}，已導入 Isotonic Regression 後處理")
        print(f"  校準後｜Brier Score={diag_after['brier_score']:.4f}　"
              f"（{'改善' if diag_after['brier_score'] < diag_before['brier_score'] else '未改善，建議檢查資料量或門檻設定'}）")
    else:
        print(f"  → 校準集 ECE 未超過門檻 {ece_threshold}，模型原生機率已足夠可靠，不套用額外校準")

    plot_path = out_dir / "calibration_curve.png"
    plot_calibration(diag_before, diag_after, str(plot_path))
    print(f"\n[已輸出] 可靠度曲線圖：{plot_path}")

    # ---- SHAP 全域重要性圖 ----
    explainer = build_shap_explainer(model, X_tr)
    shap_values_all = explainer.shap_values(X_test)
    plt.figure()
    shap.summary_plot(shap_values_all, X_test, show=False, max_display=len(all_features))  # 顯示全部特徵，不截斷
    _force_cjk_font_on_all_text(plt.gcf())
    shap_plot_path = out_dir / "shap_summary.png"
    plt.tight_layout()
    plt.savefig(shap_plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[已輸出] SHAP 全域特徵重要性圖：{shap_plot_path}")

    # ---- 特徵相關係數熱力圖 ----
    heatmap_path = out_dir / "feature_correlation_heatmap.png"
    plot_feature_correlation_heatmap(X, str(heatmap_path))
    print(f"[已輸出] 特徵相關係數熱力圖：{heatmap_path}")

    # ---- 特徵影響程度長條圖（平均 |SHAP value|）----
    importance_bar_path = out_dir / "shap_importance_bar.png"
    plot_shap_importance_bar(shap_values_all, all_features, str(importance_bar_path))
    print(f"[已輸出] 特徵影響程度長條圖：{importance_bar_path}")

    # ---- 針對單一專案輸出成功機率 + 特徵貢獻 ----
    x_row = X_test.iloc[[project_index]]
    result = explain_project(explainer, model, iso, x_row, all_features, dimension_map)
    result_path = out_dir / "single_project_report.json"
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"[已輸出] 單一專案預測報告：{result_path}\n")

    print("=" * 55)
    print(f" 🎯 測試集第 {project_index} 筆專案　最終成功機率：{result['final_calibrated_probability']}%"
          f"　｜　評分：{result['score_100']} / 100　（{result['grade']}）")
    print(f"    （模型原始機率 {result['raw_model_probability']}%，"
          f"{'已' if applied else '未'}套用 Isotonic 校準）")
    print("=" * 55)
    for dim, pct in sorted(result["dimension_contribution_pct"].items(),
                            key=lambda t: abs(t[1]), reverse=True):
        print(f"  {dim:<22} 貢獻 {pct:+.1f}%")
    print("\n  Top 特徵貢獻：")
    for feat, pct in list(result["feature_contribution_pct"].items())[:6]:
        print(f"    - {feat:<14} {pct:+.1f}%")

    # ---- 存成一個模型檔，之後不用重新訓練就能直接拿來預測新專案 ----
    model_path = out_dir / "model_bundle.joblib"
    save_model_bundle(model_path, model, iso, all_features, dimension_map, target_col)
    print(f"[已輸出] 模型檔（可直接載入預測新專案）：{model_path}\n")

    return {
        "model": model,
        "isotonic": iso,
        "explainer": explainer,
        "diag_test_before": diag_before,
        "diag_test_after": diag_after,
        "single_project_result": result,
        "model_path": str(model_path),
    }


# 註：命令列版本的 main()/argparse 在 notebook 裡不需要，
# 這裡直接在下面的 cell 手動呼叫 run_pipeline(...)。


## 7. 執行完整流程

In [ ]:
result = run_pipeline(
    csv_path=OUT_PATH,
    target_col=TARGET_COL,
    output_dir=OUTPUT_DIR,
    project_index=PROJECT_INDEX,
    ece_threshold=ECE_THRESHOLD,
)

## 8. 檢視圖表

In [ ]:
from IPython.display import Image, display

print("可靠度曲線 (Calibration Curve)：")
display(Image(filename=os.path.join(OUTPUT_DIR, "calibration_curve.png")))

print("SHAP 全域特徵重要性：")
display(Image(filename=os.path.join(OUTPUT_DIR, "shap_summary.png")))

print("特徵影響程度排行 (Mean |SHAP value|)：")
display(Image(filename=os.path.join(OUTPUT_DIR, "shap_importance_bar.png")))

print("特徵相關係數熱力圖：")
display(Image(filename=os.path.join(OUTPUT_DIR, "feature_correlation_heatmap.png")))

## 9. 檢視單一專案報告

In [ ]:
import json
with open(os.path.join(OUTPUT_DIR, "single_project_report.json"), encoding="utf-8") as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

## 10. 模型檔

`run_pipeline(...)` 已經把訓練好的模型存成 `model_bundle.joblib`（在 `OUTPUT_DIR` 底下），
裡面包含模型本身、Isotonic 校準器（如果有套用的話）、特徵清單、維度對應表。
之後不用重新跑一次訓練，直接載入這個檔案就能對新專案評分。

In [ ]:
print(f"模型檔位置：{result['model_path']}")

## 11. `predict.py` —— 對新專案評分 / 小型真實測試

- **純預測**：只知道特徵、不知道實際結果（例如評估一個還在募資中的新專案）
- **小型真實測試**：這幾筆專案已經知道實際成功/失敗，想驗證模型準不準——把 `target_col` 設成
  你的標籤欄位名稱，會多印出這批小樣本的 AUC / Brier Score，方便肉眼核對每一筆的預測是否準確

新資料不需要跟訓練時欄位完全一致：缺的特徵欄位自動補 NaN，多的欄位自動忽略。

In [ ]:
# -*- coding: utf-8 -*-
"""
predict.py
==========
載入 crowdfunding_model.py 訓練完存下來的模型檔（model_bundle.joblib），
對「新的」或「留出來沒拿去訓練」的真實專案評分——這就是「小型真實測試」的工具。

兩種用法：

1) 純預測（不知道實際結果，例如評估一個還在募資中的新專案）：
   python predict.py --model model_bundle.joblib --csv new_projects.csv

2) 小型真實測試（已經知道這幾筆專案實際成功/失敗，想驗證模型準不準）：
   python predict.py --model model_bundle.joblib --csv holdout_projects.csv --target label
   （多加 --target 之後，會額外印出這批小樣本的 AUC / Brier Score，
     並逐筆列出「預測機率 vs 實際結果」方便肉眼核對）

輸入的 CSV 不需要跟訓練時欄位完全一致：
  - 缺少的特徵欄位會自動補 NaN（交給 XGBoost 原生處理，就跟訓練時處理跨類別缺值一樣）
  - 多出來的欄位會被忽略
  - 如果有 project_id 欄位，輸出時會一併帶著方便對照
"""

import argparse
import json

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import brier_score_loss, roc_auc_score

def score_new_projects(model_path: str, csv_path: str, target_col: str | None = None,
                        id_col: str = "project_id", top_k_features: int = 5):
    bundle = load_model_bundle(model_path)
    model = bundle["model"]
    iso = bundle["isotonic"]
    all_features = bundle["all_features"]
    dimension_map = bundle["dimension_map"]

    df_new = pd.read_csv(csv_path)
    missing_cols = [f for f in all_features if f not in df_new.columns]
    if missing_cols:
        print(f"  ⚠️ 這批新資料缺少 {len(missing_cols)} 個訓練時用過的特徵欄位，"
              f"會以 NaN 補上（不會用其他值去猜）：{missing_cols}")

    ids = df_new[id_col] if id_col in df_new.columns else pd.Series(range(len(df_new)), name="row_index")
    X_new = df_new.reindex(columns=all_features)  # 缺的欄位自動變 NaN，多的欄位自動捨棄

    raw_prob = model.predict_proba(X_new)[:, 1]
    final_prob = iso.transform(raw_prob) if iso is not None else raw_prob

    explainer = build_shap_explainer(model, X_new)
    results = []
    for i in range(len(X_new)):
        row_result = explain_project(explainer, model, iso, X_new.iloc[[i]], all_features, dimension_map)
        top_feats = list(row_result["feature_contribution_pct"].items())[:top_k_features]
        results.append({
            "project_id": ids.iloc[i],
            "final_probability_pct": row_result["final_calibrated_probability"],
            "top_feature_contributions_pct": dict(top_feats),
        })

    out_df = pd.DataFrame([{
        "project_id": r["project_id"],
        "final_probability_pct": r["final_probability_pct"],
    } for r in results])

    if target_col and target_col in df_new.columns:
        y_true = df_new[target_col].astype(int).values
        out_df["actual_label"] = y_true
        out_df["correct_direction"] = (
            (out_df["final_probability_pct"] >= 50) == (y_true == 1)
        )
        auc = roc_auc_score(y_true, final_prob) if len(set(y_true)) > 1 else float("nan")
        brier = brier_score_loss(y_true, final_prob / 100 if final_prob.max() > 1 else final_prob)
        print(f"\n[小型真實測試結果]（{len(df_new)} 筆）")
        print(f"  AUC = {auc:.4f}" if not np.isnan(auc) else "  AUC 無法計算（這批資料只有單一類別）")
        print(f"  Brier Score = {brier:.4f}")
        print(f"  方向判斷正確率（機率≥50%視為預測成功）= {out_df['correct_direction'].mean():.1%}")

    return out_df, results


# 註：命令列版本的 main()/argparse 在 notebook 裡不需要，
# 這裡直接在下面的 cell 手動呼叫 score_new_projects(...)。


## 12. 執行小型真實測試

把 `TEST_CSV` 換成你要測試的一小批真實專案（例如最近剛結束、還沒被拿去訓練的專案）。
如果這批資料裡有實際結果欄位，把 `TEST_TARGET_COL` 設成該欄位名稱；
不知道實際結果的話就設成 `None`，只會印出預測機率。

In [ ]:
TEST_CSV = os.path.join(OUTPUT_DIR, "merged_training_data.csv")  # 換成你的小型真實測試 CSV
TEST_TARGET_COL = "label"  # 不知道實際結果的話改成 None

out_df, results = score_new_projects(
    model_path=result["model_path"],
    csv_path=TEST_CSV,
    target_col=TEST_TARGET_COL,
)
out_df.head(10)